<a href="https://colab.research.google.com/github/sangram-jr/Deep-Learning/blob/main/Next-word-prediction/Next_Word_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [7]:
df = pd.read_csv("qoute_dataset.csv")

In [8]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [9]:
df.shape

(3038, 2)

In [10]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [11]:

# convert all to lower case
quotes = quotes.str.lower()

# remove punctuation
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))


In [12]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


# Tokenization

In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [14]:
vocab_size = 10000

tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)


sequence = tokinizer.texts_to_sequences(quotes)



In [15]:


for i in range(3):
  print(quotes[i])



“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [16]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


set input and output

In [17]:
#One sentence
#     ↓
#[Word1, Word2, Word3, Word4]
#     ↓     X                    y
#     ├── [Word1]              → Word2
#     ├── [Word1, Word2]       → Word3
#     └── [Word1, Word2, Word3] → Word4

X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)




In [18]:
max_len = max(len(x) for x in X)
print(max_len)

745


add padding to input

In [19]:
# make all your input sequences the same length
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [20]:
y = np.array(y)

In [21]:
X_padded.shape

(85271, 745)

In [22]:
#Each target word will be represented using a vector of 2,000 positions using one hot encoding

from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)



In [23]:
y.shape

(85271,)

In [24]:
y_one_hot.shape

(85271, 10000)

# simple RNN model

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense



embedding_dim = 50
rnn_units = 128 ##RNN will maintain a hidden state of size:

rnn_model = Sequential()

#add embedding to rnn_model
rnn_model.add(
    Embedding(
        input_dim=vocab_size, #The model has a vocabulary of up to 2,000 word IDs.
        output_dim=embedding_dim, #Each word is converted into a 16-dimensional vector.
        input_length=max_len
    )
)

#Add SimpleRNN
rnn_model.add(SimpleRNN(units=rnn_units))
#output layer(creates 2,000 output neurons)
rnn_model.add(Dense(units=vocab_size, activation='softmax')) #Softmax converts the 2,000 outputs into probabilities.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [26]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [27]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# LSTM model

In [28]:
lstm_model = Sequential()

lstm_model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_len
       )
)

lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [29]:


lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)



In [30]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Train the LSTM model

In [36]:
epochs=100
batch_size=128


In [37]:
history_lstm=lstm_model.fit(
    X_padded,
    y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 35s 58ms/step - accuracy: 0.1640 - loss: 4.6345 - val_accuracy: 0.1163 - val_loss: 6.7472
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1710 - loss: 4.5053 - val_accuracy: 0.1160 - val_loss: 6.8384
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 34s 56ms/step - accuracy: 0.1804 - loss: 4.3854 - val_accuracy: 0.1161 - val_loss: 6.9407
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1930 - loss: 4.2680 - val_accuracy: 0.1155 - val_loss: 7.0232
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.2057 - loss: 4.1560 - val_accuracy: 0.1125 - val_loss: 7.1112
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.2204 - loss: 4.0495 - val_accuracy: 0.1136 - val_loss: 7.2012
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.2354 - loss: 3.9484 - val_accuracy: 0.1152 - val_loss: 7.2743
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.2490 - loss: 3

In [39]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.h5")

In [40]:
lstm_model.save('lstm_model.h5')

# prediction

In [43]:
#It converts: (word → number) into (number → word)

index_to_word = {}
for word, index in tokinizer.word_index.items():
  index_to_word[index] = word

In [44]:
def predictor(model,tokenizer,text,max_len):
  #Convert text to lowercase
  text = text.lower()
  # Convert text into word IDs
  seq = tokenizer.texts_to_sequences([text])[0]
  # add padding into text because model expects every input to have the same length.
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')
  # Make prediction
  pred = model.predict(seq,verbose = 0)
  #Find the highest-probability word
  #np.argmax() finds the position/index of the largest value.
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]




In [45]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokinizer,seed_text,max_len)
print(next_word)

implying


In [64]:
#It predicts one word at a time, adds that word to the predictor fucntion, then uses the new sentence to predict the next word.
def generate_text(model,tokenizer,seed_text,max_len,n_words): #n_words => Number of new words to generate
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [65]:
seed = "i love"
generate_text = generate_text(lstm_model,tokinizer,seed,max_len,5)
print(generate_text)



i love you in this way because


download tokenizer,max_len,index_to_word

In [66]:
import pickle

#Save tokenizer
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokinizer, f)

#Save max_len
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)

#Save index → word mapping
with open("index_to_word.pkl", "wb") as f:
  pickle.dump(index_to_word, f)
